<a href="https://colab.research.google.com/github/gabiuxo/Algoritmos-de-Aprendizaje-Automatico/blob/main/PracticaLunesTema13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctica lunes - Tema 13: Serialización de modelos y APIs

**Gabriel Elizondo Martinez**  
**Matrícula:** AL07009102


## Reto 1. Entrenamiento y serialización del pipeline completo con Joblib


In [1]:
from pathlib import Path

from joblib import dump, load
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# create the base data and split it
X, y = make_classification(n_samples=1000, n_features=4, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# train the complete preprocessing and classification pipeline
pipe_churn = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe_churn.fit(X_train, y_train)

# save and verify the serialized artifact
model_path = Path("modelos/v1/model.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)
dump(pipe_churn, model_path)

pipe_loaded = load(model_path)
offline_accuracy = float(pipe_loaded.score(X_test, y_test))

print(f"Model saved: {model_path.exists()}")
print(f"Test accuracy after loading: {offline_accuracy:.4f}")


Model saved: True
Test accuracy after loading: 0.8850


Entrené un solo pipeline con el escalado y la regresión logística. Lo guardé con Joblib y después lo cargué de nuevo para comprobar que el archivo puede usarse sin volver a entrenar el modelo.


## Reto 2. Registro de metadatos de entrenamiento


In [2]:
import json
import platform
import sklearn
import joblib

# register the context required to identify this model version
metadata = {
    "model_name": "classification_pipeline",
    "model_version": "v1",
    "metric_offline": "accuracy",
    "metric_value": offline_accuracy,
    "libraries": {
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__,
        "python": platform.python_version(),
    },
}

metadata_path = model_path.with_name("metadata.json")
metadata_path.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print(metadata_path.read_text(encoding="utf-8"))


{
  "model_name": "classification_pipeline",
  "model_version": "v1",
  "metric_offline": "accuracy",
  "metric_value": 0.885,
  "libraries": {
    "scikit_learn": "1.6.1",
    "joblib": "1.6.0",
    "python": "3.13.15"
  }
}


Guardé los datos básicos para identificar el experimento: nombre, versión, métrica y librerías. Esto mantiene el contexto del archivo serializado y permite saber con qué configuración se produjo.


## Reto 3. Contrato de entrada con Pydantic


In [3]:
from pydantic import BaseModel


class PredictionInput(BaseModel):
    feature_1: float
    feature_2: float
    feature_3: float
    feature_4: float


# validate one payload before sending it to the model
example_input = PredictionInput(
    feature_1=0.15,
    feature_2=-1.20,
    feature_3=0.80,
    feature_4=2.10,
)

validated_input = (
    example_input.model_dump()
    if hasattr(example_input, "model_dump")
    else example_input.dict()
)
print(validated_input)


{'feature_1': 0.15, 'feature_2': -1.2, 'feature_3': 0.8, 'feature_4': 2.1}


Definí cuatro entradas numéricas porque el pipeline fue entrenado con cuatro características. La validación obliga a recibir esa estructura y evita que datos faltantes o con tipos incorrectos lleguen silenciosamente a la inferencia.


## Reto 4. Endpoint de inferencia y umbral operativo


In [4]:
# run this line in Google Colab only if FastAPI is not installed
# !pip -q install fastapi uvicorn

import pandas as pd
from fastapi import FastAPI

# load the model once when the service starts
model_service = load(model_path)
app = FastAPI(title="Classification API", version=metadata["model_version"])
threshold = 0.50


@app.post("/predict")
def predict(payload: PredictionInput):
    data = payload.model_dump() if hasattr(payload, "model_dump") else payload.dict()
    X_new = pd.DataFrame([data])
    probability = float(model_service.predict_proba(X_new)[0, 1])

    return {
        "model_version": metadata["model_version"],
        "probability": probability,
        "prediction": int(probability >= threshold),
        "threshold": threshold,
    }


# test the endpoint logic with the validated example
print(predict(example_input))


{'model_version': 'v1', 'probability': 0.2536639840962325, 'prediction': 0, 'threshold': 0.5}


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


Cargué el modelo una sola vez, antes del endpoint, para no cargar el archivo en cada solicitud. El endpoint devuelve tanto la probabilidad como la clase usando un umbral de 0.50; así la decisión puede ajustarse después sin modificar el modelo entrenado.


## Conclusión

En esta práctica comprobé que un modelo no debe guardarse aislado: necesita su pipeline, metadatos y un contrato de entrada claro. Al devolver la probabilidad junto con la decisión, puedo mantener la API trazable y ajustar el umbral según la necesidad operativa.
